### Make dict with item url being the (normalized) URL, and the value being the bibtex citekey

Used for matching URL cites in perplexity dialogs to the bibext citekey filenames of obsidian notes.

In [ ]:
TODO: 
- SPACE BEFORE LINKS IN MD DOC, 
- SOME UNCLOSED '']'' NEAR eol
- ORIG FOOTNOTES ALSO MISSING
- RETAIN ORIGINAL CONTENTS, SO CAN CONSIDER ADDING NEW LINKS FROM IT LATER

In [6]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pathlib as pl
import sys

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
tmp_dir = rfw.refwrangle_test_dir / 'tmp'

perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
obsidian_citekeys_file = rfw.refwrangle_test_dir / "dat" / 'obsnotecitekeys.csv'

output_file = tmp_dir / "tmp_perplex_dialog_with_files.md"

In [2]:
zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
parentItems = zot.everything(zot.top())

In [11]:
citekeys = {}
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
for (key, value_list) in citekeysForURL.items():
    url_to_citekey[key] = value_list[0]




In [12]:
rfw.replace_perplexity_citations(perplexity_dialog_file, url_to_citekey, output_file)